In [1]:
import pandas as pd
import requests
import time
import re
from datetime import datetime

# Caminho do arquivo .xls
arquivo = r"D:\Cotic\CONFERE_CUSTODIADOS_25_02_2026 16_50_28.xls"

# Lê a planilha
df = pd.read_excel(arquivo, sheet_name="confereCustodiado", engine="xlrd")
ids_raw = df.iloc[:, 0].astype(str).str.strip()

# Normaliza os IDs
ids = []
for val in ids_raw:
    if val and val.strip():
        try:
            ids.append(int(val))
        except ValueError:
            print(f"Valor não convertido: {val}")

print(f"Total de IDs capturados: {len(ids)}")

# Login para obter token (XML)
login_url = "http://172.18.4.46:8080/sispen_api/login/login"
login_payload = {"username": "elisangela.helcias", "password": "30069218"}
headers_login = {"Content-Type": "application/json"}

resp_login = requests.post(login_url, json=login_payload, headers=headers_login)
if resp_login.status_code == 200:
    match = re.search(r"<token>(.*?)</token>", resp_login.text)
    if match:
        token = match.group(1)
        TOKEN = f"Bearer {token}"
    else:
        raise Exception("Token não encontrado na resposta XML")
else:
    raise Exception(f"Falha no login: {resp_login.status_code}")

headers = {"Authorization": TOKEN, "Accept": "application/json"}

resultados = []

def get_data_entrada(payload_ficha, headers, tentativas=3, delay=2):
    for i in range(tentativas):
        resp_ficha = requests.post(url_ficha, headers=headers, json=payload_ficha)
        if resp_ficha.status_code == 200:
            ficha = resp_ficha.json()
            custodiado = ficha.get("custodiado", {})
            
            # tenta direto
            data_entrada = custodiado.get("dataUltimaEntrada")
            if data_entrada:
                return data_entrada

            # tenta dataAtualEntradaSistemaPrisional
            epoch = custodiado.get("dataAtualEntradaSistemaPrisional")
            if epoch:
                return datetime.fromtimestamp(epoch/1000).strftime("%d/%m/%Y %H:%M:%S")

            # tenta movimentações
            movimentacoes = custodiado.get("movimentacoes", [])
            entradas = [
                m for m in movimentacoes
                if m.get("tipoMovimento", {}).get("descricao") == "ENTRADA"
                and m.get("situacao", {}).get("descricao") == "RECOLHIDO"
            ]
            if entradas:
                ultima = max(entradas, key=lambda x: x.get("dataAtualizacao") or "")
                return ultima.get("dataMovimentacao")

        time.sleep(delay)
    return None


# Loop pelos IDs
for pessoa_id in ids:
    try:
        # 1. Obter id_real
        url_real = f"http://172.18.4.46:8080/sispen_api/custodiado?page=0&size=10&ativo=true&unidades=170&pessoaId={pessoa_id}"
        resp_real = requests.get(url_real, headers=headers)

        if resp_real.status_code == 200:
            data_real = resp_real.json()
            entities_real = data_real.get("entities", [])
            if not entities_real:
                print(f"{pessoa_id} - Não encontrado no endpoint custodiado")
                continue

            id_real = entities_real[0]["id"]

            # 2. Buscar visitantes
            url_visita = f"http://172.18.4.46:8080/sispen_api/fichas-custodiados/{id_real}/visitantes?page=0&size=32&sort=pessoa.nome%2Casc"
            resp_visita = requests.get(url_visita, headers=headers)

            nome_visitante = "Sem cadastro"
            relacao = ""
            ultima_visita = None
            id_visitante_pessoa = None

            if resp_visita.status_code == 200:
                data = resp_visita.json()
                entities = data.get("entities", [])
                if entities:
                    visitantes_validos = [v for v in entities if v.get("dataUltimaVisita")]
                    if visitantes_validos:
                        ultimo = max(visitantes_validos, key=lambda x: x["dataUltimaVisita"])
                        nome_visitante = ultimo["pessoa"]["nome"].strip()
                        relacao = ultimo.get("tipoPessoaRelacao", {}).get("descricao", "")
                        data_epoch = ultimo["dataUltimaVisita"]
                        ultima_visita = datetime.fromtimestamp(data_epoch / 1000).strftime("%d/%m/%Y")
                        id_visitante_pessoa = ultimo["pessoa"]["id"]
                    else:
                        primeiro = entities[0]
                        nome_visitante = primeiro["pessoa"]["nome"].strip()
                        relacao = primeiro.get("tipoPessoaRelacao", {}).get("descricao", "")
                        ultima_visita = "Sem data de visita"
                        id_visitante_pessoa = primeiro["pessoa"]["id"]

                # calcula status da visita
                visita_status = ""
                if ultima_visita and ultima_visita != "Sem data de visita":
                    try:
                        data_visita = datetime.strptime(ultima_visita, "%d/%m/%Y")
                        diferenca = datetime.now() - data_visita
                        if diferenca.days > 90:   # aproximadamente 3 meses
                            visita_status = "Mais de 3 meses"
                        else:
                            visita_status = "Menos de 3 meses"
                    except:
                        visita_status = "Data inválida"
                else:
                    visita_status = "Sem registro"


            # 3. Buscar ficha do custodiado (dataUltimaEntrada)
            pessoa_obj = entities_real[0].get("pessoa", {})
            id_pessoa_ficha = pessoa_obj.get("id", pessoa_id)  # usa o id da pessoa se disponível, senão cai no da planilha
            url_ficha = "http://172.18.4.46:8080/sispen_api/custodiado/ficha"
            payload_ficha = {
                "idPessoa": str(id_pessoa_ficha),   # <-- aqui está a mudança
                "idUsuario": 3104,
                "idUnidade": 170,
                "idOrigem": 1,
                "unidadesAcesso": [170,157]
            }
            data_entrada = get_data_entrada(payload_ficha, headers)
            if data_entrada:
                try:
                    data_legivel = datetime.strptime(data_entrada, "%Y-%m-%d %H:%M:%S").strftime("%d/%m/%Y %H:%M:%S")
                except:
                    data_legivel = data_entrada
            else:
                data_legivel = "Sem registro"

            # calcula status da entrada
            entrada_status = ""
            if data_legivel and data_legivel != "Sem registro":
                try:
                    data_entrada_dt = datetime.strptime(data_legivel, "%d/%m/%Y %H:%M:%S")
                    diferenca = datetime.now() - data_entrada_dt
                    if diferenca.days > 90:   # mais de 3 meses
                        entrada_status = "Mais de 3 meses"
                    else:
                        entrada_status = "Menos de 3 meses"
                except:
                    entrada_status = "Data inválida"
            else:
                entrada_status = "Sem registro"


            
            # 4. Buscar ficha do visitante (telefone)
            contato = ""
            if id_visitante_pessoa:
                url_ficha_visitante = "http://172.18.4.46:8080/sispen_api/visitante/obterficha"
                payload_visitante = {
                    "idPessoa": str(id_visitante_pessoa),
                    "idUsuario": 3104,
                    "idUnidade": 170,
                    "idOrigem": 1,
                    "unidadesAcesso": [170,157]
                }
                resp_ficha_visitante = requests.post(url_ficha_visitante, headers=headers, json=payload_visitante)
                if resp_ficha_visitante.status_code == 200:
                    ficha_v = resp_ficha_visitante.json()
                    telefones = ficha_v.get("pessoa", {}).get("telefones", [])
                    if telefones:
                        primeiro_tel = telefones[0]
                        contato = f"{primeiro_tel.get('ddd','')} {primeiro_tel.get('telefone','')}"
            # 5. Buscar último atendimento de Serviço Social
            url_atendimento = f"http://172.18.4.46:8080/sispen_api/atendimentomedico/paginado?page=0&size=10&sort=dataAtendimento%2Cdesc&idCustodiado={id_real}"
            resp_atendimento = requests.get(url_atendimento, headers=headers)
            descricao_social = ""
            data_social_legivel = ""
            if resp_atendimento.status_code == 200:
                atendimentos = resp_atendimento.json().get("entities", [])
                for at in atendimentos:
                    tipo = at.get("tipoEspecialidade", {}).get("descricao", "")
                    if tipo == "SERVIÇO SOCIAL":
                        descricao_social = at.get("descricao", "")
                        data_epoch = at.get("dataAtendimento")
                        if data_epoch:
                            data_social_legivel = datetime.fromtimestamp(data_epoch / 1000).strftime("%d/%m/%Y")
                        break

            
            # 6. Saída final
            if ultima_visita:
                print(f"{pessoa_id} (real: {id_real}) - {nome_visitante} - {relacao} - {contato} - Última visita: {ultima_visita} - Última entrada: {data_legivel} - Serviço Social: {descricao_social} ({data_social_legivel})")
            else:
                print(f"{pessoa_id} (real: {id_real}) - {nome_visitante} - {relacao} - {contato} - Sem data de visita - Última entrada: {data_legivel} - Serviço Social: {descricao_social} ({data_social_legivel})")

        else:
            print(f"{pessoa_id} - Erro custodiado {resp_real.status_code}")
        
        # monta linha para planilha (sempre executa, independente do if/else)
        resultados.append({ 
            "Prontuário": pessoa_id, 
            "Nome": df.loc[df.iloc[:,0] == pessoa_id].iloc[0,1] if 1 < df.shape[1] else "", 
            "Ult. Localização": df.loc[df.iloc[:,0] == pessoa_id].iloc[0,2] if 2 < df.shape[1] else "", 
            "Ult. Visitante": nome_visitante, 
            "Rel. visitante": relacao, 
            "Contato Vis.": contato, 
            "Data Ult. Visita": ultima_visita, 
            "Data Entrada Unid.": data_entrada, 
            "Ult. Obs. do Social": f"{descricao_social} ({data_social_legivel})",
            "Visita": visita_status,
            "Entrada": entrada_status 
        })

    except Exception as e:
        print(f"{pessoa_id} - Erro na requisição: {e}")

    time.sleep(1)

# cria DataFrame final 
df_final = pd.DataFrame(resultados) 

# salva em Excel 
saida = r"D:\Cotic\resultado_confereCustodiados.xlsx" 
df_final.to_excel(saida, index=False) 
print(f"Planilha gerada em: {saida}")

Total de IDs capturados: 187
303990 (real: 150046) - MARIA MARLIETE GRAÇA MARROCOS - COMPANHEIRO - 88 981097738 - Última visita: 31/01/2026 - Última entrada: 04/09/2024 23:21:25 - Serviço Social: Triagem realizada na ala I, na data de 16/01/2026, acompanhada de Policial , interno não apresentou demanda ao serviço social. (Mayara Ximenes - assistente social cress 8723). (27/01/2026)
306041 (real: 150868) - AURILENE DOS SANTOS NASCIMENTO - IRMÃO - 85 991617936 - Última visita: 14/02/2026 - Última entrada: 06/02/2024 15:35:18 - Serviço Social: Triagem realizada na data de 15.01.2026, interno não apresentou demanda ao serviço social. (Mayara Ximenes - assistente social cress 8723). (26/01/2026)
445909 (real: 204831) - ANTONIO VILEMAR DA SILVA - IRMÃO - 88 996348729 - Última visita: 17/01/2026 - Última entrada: 10/08/2021 17:04:33 - Serviço Social: Triagem realizada na ala I, na data de 16/01/2026, acompanhada de Policial , interno não apresentou demanda ao serviço social. (Mayara Ximenes -